In [0]:
%run ./99_Config

In [0]:
%run ./98_Utilities

In [0]:
%run ./97_Logger

In [0]:
from pyspark.sql.functions import *

pipeline_start("05_Bronze_to_Silver")

# COMMAND ----------
validated_root = VALIDATED_PATH
silver_root = SILVER_PATH

datasets = sorted([
    f.name.replace("/", "")
    for f in dbutils.fs.ls(validated_root)
])

print("Validated datasets:", datasets)

In [0]:
total_rows = 0
processed = 0

for dataset in datasets:

    validated_path = f"{validated_root}/{dataset}"
    silver_path = f"{silver_root}/{dataset}"

    print_header(f"Transforming {dataset}")

    try:

        df = read_delta(validated_path)

        rows = record_count(df)

        # Generic transformations
        for c in df.columns:
            if dict(df.dtypes)[c] == "string":
                df = df.withColumn(c, trim(col(c)))

        if "currency" in df.columns:
            df = df.withColumn("currency", upper(col("currency")))

        if "branch" in df.columns:
            df = df.withColumn("branch", initcap(col("branch")))

        if "transaction_timestamp" in df.columns:
            df = (
                df.withColumn("transaction_date", to_date(col("transaction_timestamp")))
                  .withColumn("transaction_year", year(col("transaction_timestamp")))
                  .withColumn("transaction_month", month(col("transaction_timestamp")))
            )

        if "amount" in df.columns:
            df = (
                df.withColumn(
                    "transaction_category",
                    when(col("amount") < 1000, "LOW")
                    .when(col("amount") < 50000, "MEDIUM")
                    .otherwise("HIGH")
                )
            )

        df = (
            df.withColumn("silver_load_timestamp", current_timestamp())
              .withColumn("pipeline_stage", lit("SILVER"))
        )

        write_delta(df, silver_path, "overwrite")
        create_delta_table(f"silver_{dataset}", silver_path)

        audit(
            notebook="05_Bronze_to_Silver",
            dataset=dataset,
            rows_read=rows,
            rows_written=rows,
            rejected_rows=0,
            status="SUCCESS"
        )

        log_success(
            notebook="05_Bronze_to_Silver",
            dataset=dataset,
            rows_read=rows,
            rows_written=rows,
            execution_time="Completed"
        )

        display_summary(dataset, rows, rows, 0)

        processed += 1
        total_rows += rows

    except Exception as ex:

        log_error(
            notebook="05_Bronze_to_Silver",
            dataset=dataset,
            exception=str(ex)
        )

        audit(
            notebook="05_Bronze_to_Silver",
            dataset=dataset,
            rows_read=0,
            rows_written=0,
            rejected_rows=0,
            status="FAILED",
            error=str(ex)
        )

In [0]:
print("="*80)
print("BRONZE TO SILVER SUMMARY")
print("="*80)
print(f"Datasets Processed : {processed}")
print(f"Rows Written       : {total_rows}")
print("="*80)

pipeline_end("05_Bronze_to_Silver")